# Role A: EchoNext loader and baselines

Attach your existing private EchoNext dataset to this private Kaggle notebook. Enable Internet for the GitHub checkout and public Mini weights. The notebook uses Kaggle's preinstalled Python packages. CPU works for both baselines. A GPU can speed up Mini inference, while gradient boosting uses CPU.
In **Add-ons → Secrets**, add `GITHUB_TOKEN` with a GitHub personal access token that can read `andykimdev/compress_ECG`, and grant this notebook access to the secret.

Run `smoke` first on up to 1,000 validation ECGs. Then switch to `full` for all validation and test ECGs. Full runs save aggregate metrics separately for the repository.


In [ ]:
from base64 import b64encode
from pathlib import Path
import os
import subprocess
import sys

BRANCH = 'shaurya/role-a-baselines'  # Push this branch to GitHub before running.
CODE_COMMIT = None                 # Optional commit SHA from this branch.
REPO_DIR = Path('/tmp/compress_ECG_role_a')
REPO_URL = 'https://github.com/andykimdev/compress_ECG.git'

def clone_private_repo():
    from kaggle_secrets import UserSecretsClient

    try:
        token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
    except Exception:
        raise RuntimeError('Add GITHUB_TOKEN in Kaggle Add-ons → Secrets and grant this notebook access.') from None
    if not token:
        raise RuntimeError('GITHUB_TOKEN is empty. Update it in Kaggle Add-ons → Secrets.')

    credentials = b64encode(f'x-access-token:{token}'.encode()).decode()
    # Pass the header only to Git, with the plain URL recorded as origin.
    clone_env = dict(os.environ, GIT_TERMINAL_PROMPT='0',
                     ECHONEXT_GIT_AUTH_HEADER=f'Authorization: Basic {credentials}')
    try:
        subprocess.run([
            'git', f'--config-env=http.{REPO_URL}.extraHeader=ECHONEXT_GIT_AUTH_HEADER',
            'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO_DIR),
        ], env=clone_env, check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError('Git clone failed. Check Kaggle Internet, token access to the repo, and the pushed branch.') from None

if not (REPO_DIR / '.git').is_dir():
    clone_private_repo()
if CODE_COMMIT is not None:
    subprocess.run(['git', 'checkout', CODE_COMMIT], cwd=REPO_DIR, check=True)

try:
    import torch
except ImportError as error:
    raise RuntimeError('Kaggle could not import PyTorch. Stop this session and start a fresh one before running setup again.') from error

# Import directly from the checkout so setup leaves Kaggle's PyTorch installation alone.
sys.path.insert(0, str(REPO_DIR / 'src'))
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('Code commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())


## Choose the data and run mode

`DATA_DIR` is the folder containing the metadata CSV and the split arrays. The notebook can discover it when exactly one copy is attached. Use a fresh `RUNS_ROOT` to repeat a completed run in the same Kaggle session.


In [ ]:
from compress_ecg.data import METADATA_FILENAMES

DATA_DIR = None  # Set Path('/kaggle/input/...') if auto-discovery finds several copies.
RUN_MODE = 'smoke'  # Change to 'full' after the first run passes.
if DATA_DIR is None:
    matches = {path.parent for name in METADATA_FILENAMES for path in Path('/kaggle/input').rglob(name)}
    if len(matches) != 1:
        raise ValueError(f'Set DATA_DIR to the exact file directory; found {len(matches)} candidate folders')
    DATA_DIR = matches.pop()
DATA_DIR = Path(DATA_DIR)
if RUN_MODE not in ('smoke', 'full'):
    raise ValueError('Choose smoke or full')
SPLITS = ('val',) if RUN_MODE == 'smoke' else ('val', 'test')
LIMIT = 1000 if RUN_MODE == 'smoke' else None
RUNS_ROOT = Path('/kaggle/working/runs')
if RUN_MODE == 'smoke':
    RUNS_ROOT = RUNS_ROOT / 'smoke'
print('Dataset root:', DATA_DIR)
print('Output root:', RUNS_ROOT)


## Check the shared loader

A batch has waveform `[B, 12, 2500]`, tabular features `[B, 7]`, labels `[B, 12]`, and a label mask `[B, 12]`. A false mask value marks a missing target. The notebook prints shapes and counts.


In [ ]:
from torch.utils.data import DataLoader
from compress_ecg.data import EchoNextDataset, LABEL_COLUMNS, SHD_INDEX

dataset = EchoNextDataset(DATA_DIR, 'val', limit=LIMIT, seed=42)
batch = next(iter(DataLoader(dataset, batch_size=8, shuffle=False)))
print({name: tuple(batch[name].shape) for name in ('waveform', 'tabular', 'labels', 'label_mask')})
print('Selected validation ECGs:', len(dataset))
print('Observed targets per output in the batch:', batch['label_mask'].sum(0).tolist())


## Run Mini and train gradient boosting

The Mini downloader verifies the released architecture and weights. Gradient boosting fits one classifier per label on the official train split using the seven tabular features. Each scoring run records its inputs and outputs. The 0.820 SHD AUROC is the published test reference.


In [ ]:
from compress_ecg.baseline import run_baseline, run_gradient_boosting
from compress_ecg.predictions import run_dir

subprocess.run(['bash', 'scripts/download_weights.sh', '--mini-only'], cwd=REPO_DIR, check=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for split in SPLITS:
    for model_name in ('mini', 'gradient_boosting'):
        destination = run_dir(model_name, split, RUNS_ROOT)
        print(f'Running {model_name} on {split}...', flush=True)
        if model_name == 'mini':
            result = run_baseline(DATA_DIR, destination, split=split, limit=LIMIT, device=DEVICE)
        else:
            result = run_gradient_boosting(DATA_DIR, destination, split=split, limit=LIMIT)
        print(result['n_records'], 'ECGs | SHD:', result['metrics'][LABEL_COLUMNS[SHD_INDEX]])


## Verify and save the outputs

Save the private notebook version with outputs. For each model and split, keep `predictions.npz`, `metadata.csv`, `manifest.json`, and `metrics.json` together. Gradient boosting also saves `gradient_boosting.joblib`, the fitted checkpoint. The released Mini weights can be retrieved with the pinned download script.

The full run writes four aggregate metric JSON files under `/kaggle/working/results/baselines/`. Those files can go in the repository's `results/baselines/`. The run folders and fitted checkpoint stay private.


In [ ]:
import json
import numpy as np
from compress_ecg.predictions import load_run

for split in SPLITS:
    for model_name in ('mini', 'gradient_boosting'):
        run = load_run(run_dir(model_name, split, RUNS_ROOT))
        probabilities = run.arrays['probabilities']
        assert probabilities.shape == (run.manifest['n_records'], len(LABEL_COLUMNS))
        assert np.isfinite(probabilities).all()
        print(f'{model_name} / {split}: {probabilities.shape} finite probabilities')
        if RUN_MODE == 'full':
            results_dir = Path('/kaggle/working/results/baselines')
            results_dir.mkdir(parents=True, exist_ok=True)
            summary = {
                'model': model_name, 'split': split, 'n_records': run.manifest['n_records'],
                'code': run.manifest['code'], 'data_version': run.manifest['data_version'],
                'metrics': run.metrics,
            }
            (results_dir / f'{model_name}_{split}_metrics.json').write_text(json.dumps(summary, indent=2) + '\n')
